# Themis AI - Train on Google Colab
Trains the Themis language model on TinyStories using Colab's GPU.
Checkpoints are saved to your Google Drive so you can download them to your laptop.

**Steps:** Runtime -> Change runtime type -> GPU (T4). Then Run all.

## 1. Check the GPU Colab gave us

In [12]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

Tue Jun 23 06:51:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the project from GitHub

In [13]:
%cd /content
!rm -rf /content/themis-ai
!git clone https://github.com/jaisonkerala1/themis-ai.git
%cd /content/themis-ai


/content
Cloning into 'themis-ai'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 108 (delta 11), reused 103 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 120.22 KiB | 13.36 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/themis-ai


## 3. Install dependencies (torch is preinstalled on Colab)

In [14]:
!pip install -q datasets

## 4. Mount Google Drive (so checkpoints survive disconnects)
A popup will ask you to authorize. Checkpoints save to MyDrive/themis/.

In [15]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/themis', exist_ok=True)
print('Drive ready: /content/drive/MyDrive/themis')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready: /content/drive/MyDrive/themis


## 5. Download TinyStories (more stories than the laptop run, Colab is faster)

In [16]:
# Edit NUM_STORIES in the script for more/less. Default 2000.
# For Colab you can bump it up by editing the file, e.g. 10000.
!python scripts/prepare_tinystories.py

PREPARING TINYSTORIES DATASET

  Collected 500 stories...
  Collected 1000 stories...
  Collected 1500 stories...
  Collected 2000 stories...
  Collected 2500 stories...
  Collected 3000 stories...
  Collected 3500 stories...
  Collected 4000 stories...
  Collected 4500 stories...
  Collected 5000 stories...
  Collected 5500 stories...
  Collected 6000 stories...
  Collected 6500 stories...
  Collected 7000 stories...
  Collected 7500 stories...
  Collected 8000 stories...
  Collected 8500 stories...
  Collected 9000 stories...
  Collected 9500 stories...
  Collected 10000 stories...
  Collected 10500 stories...
  Collected 11000 stories...
  Collected 11500 stories...
  Collected 12000 stories...
  Collected 12500 stories...
  Collected 13000 stories...
  Collected 13500 stories...
  Collected 14000 stories...
  Collected 14500 stories...
  Collected 15000 stories...
  Collected 15500 stories...
  Collected 16000 stories...
  Collected 16500 stories...
  Collected 17000 stories...
  C

## 6. Train
This runs the same training as your laptop, but on Colab's faster GPU.
The checkpoint saves to the repo folder; the next cell copies it to Drive.

In [ ]:
!python scripts/train_language.py

                         LANGUAGE TRAINING (TinyStories)

Device: cuda
GPU: Tesla T4

[1] Loading TinyStories...
Loaded 20000 stories

[2] Initializing scaled model...
/content/themis-ai/themis/encoders/text_encoder.py:161: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(
Parameters: 22,464,192 (22.5M)

[3] Building next-token training pairs...
Replay buffer: 2,924,663 transitions

[4] Training...
Epochs: 30000 | seq_len: 16 | batch: 64

Google Drive detected - checkpoints will also be copied to /content/drive/MyDrive/themis

Epoch 00001/30000 | Policy(CE):  9.184 | Avg:  9.184 | Best:  9.184 | VFE:    100.0
Epoch 00200/30000 | Policy(CE):  1.835 | Avg:  2.525 | Best:  1.694 | VFE:    100.0
Epoch 00400/30000 | Policy(CE):  1.742 | Avg:  1.714 | Best:  1.401 | VFE:    100.0
Epoch 00600/30000 | Policy(CE):  1.421 | Avg:  1.515 | Best:  1.279 | VFE:    100.0
Epoch 00800/3000

## 7. Save the trained model to Google Drive

In [ ]:
import shutil, os
src = 'checkpoint_language.pt'
dst = '/content/drive/MyDrive/themis/checkpoint_language.pt'
if os.path.exists(src):
    shutil.copy(src, dst)
    print('Saved to Drive:', dst, '(', round(os.path.getsize(dst)/1e6,1), 'MB )')
else:
    print('No checkpoint found - did training finish?')

## 8. Quick test - generate some English

In [ ]:
!python scripts/chat_language.py

## Done!
Your trained model is at **MyDrive/themis/checkpoint_language.pt**.
Download it from drive.google.com and drop it into your local project folder.
Then run `scripts/chat_language.py` or `web_app.py` on your laptop.